# VirtualFit — CatVTON Try-On Server (Colab)

Hosts the **CatVTON** virtual try-on model on a free Colab **T4 GPU** and exposes it
as a FastAPI endpoint through a **cloudflared** quick tunnel. On startup it **publishes
its tunnel URL** to a small public store, so the VirtualFit backend auto-discovers it —
you never set the URL manually.

## How to run
1. **Runtime → Change runtime type → T4 GPU**, then run every cell top-to-bottom.
2. The last cell prints the URL **and publishes it**. VirtualFit picks it up on the
   next try-on automatically.
3. Re-run this notebook whenever the Colab session dies (free Colab is not persistent).
   The backend needs no changes — the new URL is published to the same store.

Endpoints: `GET /health`, `POST /tryon`. Validated on T4: ~50s/view @ 20 steps,
~110s/view @ 50 steps, peak VRAM ~5.3 GB.

In [ ]:
# 1. Confirm GPU is attached (must say Tesla T4 / CUDA True)
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available(),
      "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
# 2. Clone CatVTON (vendors detectron2 + densepose — no compile/install hell)
%cd /content
!rm -rf CatVTON
!git clone https://github.com/Zheng-Chong/CatVTON.git
%cd CatVTON

In [ ]:
# 3. Install pinned deps. IMPORTANT: run BEFORE the build cell so the correct
#    diffusers/accelerate/peft are imported (CatVTON's bleeding-edge pin conflicts
#    with Colab's accelerate; these stable pins work together).
!pip -q install "diffusers==0.31.0" "accelerate==0.34.2" "transformers==4.46.3" \
  "peft==0.13.2" "huggingface_hub>=0.25,<0.27" \
  av pycocotools cloudpickle "fvcore==0.1.5.post20221221" omegaconf==2.3.0 \
  opencv-python scikit-image fastapi "uvicorn[standard]" python-multipart 2>&1 | tail -5
print("INSTALL DONE")

In [ ]:
# 4. Build the CatVTON pipeline + AutoMasker (downloads weights ~6GB first run).
#    fp16 forced because the T4 (Turing) does not support bf16.
import os, sys, torch
sys.path.insert(0, "/content/CatVTON")
os.chdir("/content/CatVTON")
from huggingface_hub import snapshot_download

REPO = "zhengchong/CatVTON"
BASE = "booksforcharlie/stable-diffusion-inpainting"
repo_path = snapshot_download(repo_id=REPO)

from model.pipeline import CatVTONPipeline
from model.cloth_masker import AutoMasker

pipeline = CatVTONPipeline(
    base_ckpt=BASE, attn_ckpt=repo_path, attn_ckpt_version="mix",
    weight_dtype=torch.float16, use_tf32=True, device="cuda",
)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"), device="cuda",
)
print("PIPELINE + AUTOMASKER READY")

In [ ]:
# 5. FastAPI server (reuses the loaded models, lock-serialized for one T4).
import threading, base64, io, time, requests
from fastapi import FastAPI
from pydantic import BaseModel
from PIL import Image
from diffusers.image_processor import VaeImageProcessor
from utils import resize_and_crop, resize_and_padding

W, H = 768, 1024
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False,
                                   do_binarize=True, do_convert_grayscale=True)
infer_lock = threading.Lock()

def b64_to_img(s):
    if "," in s and s.strip().startswith("data:"): s = s.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(s))).convert("RGB")

def img_to_b64(img):
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

class TryOnReq(BaseModel):
    person_image: str
    cloth_image: str
    cloth_type: str = "upper"
    num_inference_steps: int = 30
    guidance_scale: float = 2.5
    seed: int = 42

app = FastAPI(title="VirtualFit CatVTON")

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0),
            "vram_gb": round(torch.cuda.mem_get_info()[1] / 1e9, 1)}

@app.post("/tryon")
def tryon(req: TryOnReq):
    t0 = time.time()
    person = resize_and_crop(b64_to_img(req.person_image), (W, H))
    cloth  = resize_and_padding(b64_to_img(req.cloth_image), (W, H))
    with infer_lock:
        mask = automasker(person, req.cloth_type)["mask"]
        mask = mask_processor.blur(mask, blur_factor=9)
        g = torch.Generator(device="cuda").manual_seed(req.seed)
        result = pipeline(image=person, condition_image=cloth, mask=mask,
                          num_inference_steps=req.num_inference_steps,
                          guidance_scale=req.guidance_scale, generator=g)[0]
    return {"image": "data:image/png;base64," + img_to_b64(result),
            "seconds": round(time.time() - t0, 1)}

def _run():
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=_run, daemon=True).start()
time.sleep(4)
print(requests.get("http://localhost:8000/health").json())
print("SERVER UP on :8000")

In [ ]:
# 6. Tunnel + auto-publish URL. Copy nothing — VirtualFit auto-discovers it.
#    DISCOVERY_URL must match backend .env TRYON_DISCOVERY_URL.
import subprocess, re, time, os, requests
DISCOVERY_URL = "https://jsonblob.com/api/jsonBlob/019ef86e-3c96-7d7f-8d36-e14639cf7546"

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True); time.sleep(2)

logf = "/content/cf.log"
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                 stdout=open(logf, "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", open(logf).read())
    if m: PUBLIC_URL = m.group(0); break

for i in range(12):  # wait for tunnel DNS to propagate
    try:
        print("health:", requests.get(PUBLIC_URL + "/health", timeout=30).json()); break
    except Exception: time.sleep(5)

# Publish so the VirtualFit backend auto-discovers this session's URL.
if PUBLIC_URL:
    try:
        requests.put(DISCOVERY_URL, headers={"Content-Type": "application/json"},
                     json={"url": PUBLIC_URL, "updated": time.strftime("%Y-%m-%d %H:%M:%S")}, timeout=20)
        print("\n==> Published. VirtualFit will use this automatically on the next try-on.")
    except Exception as e:
        print("publish failed (set URL manually via /api/tryon/server):", e)
print("==> PUBLIC_URL:", PUBLIC_URL)